# CT-RATE full download → Google Drive

Two-cell downloader as requested: **Cell 1** pulls the **entire validation split**.
2. **Cell 2** pulls the **entire training split**.

Both save NIfTI volumes to `MyDrive/ctrate/<split>/` and are **resumable**
(already-present files are skipped, so just re-run a cell after any Colab timeout).

> ⚠️ **Size / quota.** CT-RATE is huge: the **train** split is **~20 TB** of
> `.nii.gz` and **valid** is several TB. Google Drive must have enough quota, and
> a single Colab session will not finish train in one go — plan to re-run Cell 2
> repeatedly (it resumes). To grab only a subset instead, use
> `scripts/14_ctrate_download_pairs.py` (manifest-driven), or set
> `only_nifti=False` here to also pull JSON/metadata.

**Prereqs:** accept the CT-RATE license on Hugging Face and have a **read token**
(store it as a Colab secret named `HF_TOKEN`, or paste when prompted).


In [ ]:
# ===== SETUP: mount Drive, install deps, HF login, define downloader =====
# CT-RATE + this code are on the Hugging Face Hub. You must have ACCEPTED the
# CT-RATE license on HF and have a READ token.
import os, getpass, time

from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U "huggingface_hub>=0.24"

from huggingface_hub import login, snapshot_download, HfApi

# --- token: prefer a Colab secret named HF_TOKEN, else prompt ---
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste HF read token: ')
login(HF_TOKEN)

REPO_ID   = "ibrahimhamamci/CT-RATE"
# where everything lands (one folder per split). Change if you like.
DRIVE_ROOT = "/content/drive/MyDrive/ctrate"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive target:", DRIVE_ROOT)

def download_split(split, max_workers=8, only_nifti=True, include_metadata=False):
    """Download an ENTIRE CT-RATE split ('valid' or 'train') straight to Drive.

    - Grabs both the '<split>' and '<split>_fixed' folders (v1 + v2 layouts).
    - Resumable: snapshot_download skips files already present, so if the
      Colab session dies, just re-run this cell to continue.
    - only_nifti=True restricts to the .nii.gz volumes (the imaging data).
    """
    out_dir = os.path.join(DRIVE_ROOT, split)
    os.makedirs(out_dir, exist_ok=True)

    if only_nifti:
        patterns = [f"dataset/{split}/**/*.nii.gz",
                    f"dataset/{split}_fixed/**/*.nii.gz"]
    else:
        patterns = [f"dataset/{split}/**",
                    f"dataset/{split}_fixed/**"]
    if include_metadata:
        # report CSVs + DICOM-header metadata for this split (best-effort globs)
        patterns += [f"dataset/radiology_text_reports/*{split}*",
                     f"dataset/metadata/*{split}*"]

    print(f"\n>>> downloading split='{split}'  ->  {out_dir}")
    print("    patterns:", patterns)
    t0 = time.time()
    snapshot_download(
        repo_id=REPO_ID, repo_type="dataset",
        allow_patterns=patterns,
        local_dir=out_dir,
        token=HF_TOKEN,
        max_workers=max_workers,       # parallel file downloads
        tqdm_class=None,               # default progress bars
    )
    n = sum(len(fs) for _, _, fs in os.walk(out_dir))
    print(f"<<< done split='{split}'  files_on_disk={n}  "
          f"elapsed={ (time.time()-t0)/60:.1f} min")
    return out_dir


In [ ]:
# ===== CELL 1: download ALL CT-RATE VALIDATION data to Drive =====
# Validation is the smaller split (a few TB). Resumable: re-run if it drops.
download_split("valid", max_workers=8, only_nifti=True)


In [ ]:
# ===== CELL 2: download ALL CT-RATE TRAINING data to Drive =====
# WARNING: the FULL train split is ~20 TB of NIfTI. Make sure your Drive quota
# and time budget can handle it. This is resumable — re-run to continue after a
# session timeout (already-downloaded files are skipped).
download_split("train", max_workers=8, only_nifti=True)


## Notes

- **Resume:** `snapshot_download` skips files already on disk, so re-running a
  cell continues where it stopped. Safe against session timeouts.
- **Layout:** files land under `MyDrive/ctrate/<splitset/<split>[_fixed]/<patient>/<patient_scan>/*.nii.gz`,
  matching the paths `scripts/14_ctrate_download_pairs.py` expects.
- **Metadata/reports:** set `include_metadata=True` in `download_split(...)` to
  also fetch this split's report CSVs + DICOM-header metadata.
- **Everything, not per-split:** to mirror the whole `dataset/` tree at once you
  could instead call `snapshot_download(REPO_ID, repo_type="dataset", local_dir=DRIVE_ROOT)`
  with no `allow_patterns` — but that is the full ~21.3 TB.
- **Throughput:** raise `max_workers` if the runtime + Drive can keep up; lower it
  if you hit rate limits or Drive write errors.
